# 04 — Adversarial Training Cross-Dataset (XGBoost Tunggal)

**Pertanyaan inti:** apakah *adversarial training* (ala Paper 1: finite-diff gradient + FGSM + augmentasi) memperbaiki **generalisasi lintas-jaringan** — yaitu menaikkan MCC cross-dataset dari baseline T3 yang kolaps (~0)?

**Menggabungkan dua dimensi robustness sekaligus:**
1. Robustness terhadap **evasion** (perturbasi adversarial) — dimensi Paper 1.
2. Robustness terhadap **perpindahan jaringan** (CIC ↔ UNSW) — dimensi baru paper Q1.

**Metodologi (konsisten Paper 1):**
- Model **A** saja (9 fitur irisan kuat; terbukti setara Model B di T3, lebih ringkas). Label **biner** attack/normal.
- Gradien via **finite-difference central** `S = (L(x+h)-L(x-h))/(2h)`, `h=0.01` (XGBoost non-differentiable).
- Serangan **FGSM**: `x_adv = x + eps * sign(S)`.
- **Adversarial training**: `D_robust = D_clean UNION D_adv` (80:20), retrain XGBoost.
- z-score **per dataset** (Opsi 3), fit di data latih masing-masing.

**Evaluasi diperluas (arah utama: latih di CIC):**
| Kondisi uji | Makna |
|---|---|
| CIC clean | integritas (akurasi asal terjaga?) |
| CIC adversarial | robustness evasion (in-domain) |
| **UNSW clean** | **generalisasi lintas-jaringan (fokus paper Q1)** |
| UNSW adversarial | robustness gabungan (jaringan beda + evasion) |

Arah sebaliknya (latih di UNSW → uji CIC) dijalankan untuk simetri.

**Hipotesis jujur:** adversarial training FGSM menahan perturbasi KECIL, belum tentu menutup perbedaan DISTRIBUSI antar-dataset yang besar. Jika MCC cross-dataset tetap rendah, itu **temuan** (memotivasi mekanisme cross-network khusus), bukan kegagalan. Semua angka dilaporkan apa adanya.

> Jalankan di SageMaker (butuh `cleaned_100.pkl` + CSV UNSW di `../data/`).

In [ ]:
# --- Bootstrap dependency ---
import importlib, subprocess, sys
for pkg, imp in [('pandas','pandas'), ('numpy','numpy'), ('scikit-learn','sklearn'), ('xgboost','xgboost')]:
    try:
        importlib.import_module(imp)
    except ImportError:
        print(f'[setup] installing {pkg} ...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

import pickle, os, json, time
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import matthews_corrcoef, f1_score, accuracy_score, confusion_matrix
from xgboost import XGBClassifier

CIC_PKL    = '../../CICDDoS2018/data/cleaned_100.pkl'
UNSW_TRAIN = '../data/UNSW_NB15_testing-set.csv'   # 175.341 (nama tertukar -> dipakai TRAIN)
UNSW_TEST  = '../data/UNSW_NB15_training-set.csv'  #  82.332 (dipakai TEST)
OUT_JSON   = '../adversarial_cross_dataset.json'

RANDOM_SEED = 42
H_STEP      = 0.01     # step finite-difference (di ruang ter-z-score)
EPS_TRAIN   = 0.1      # epsilon utk augmentasi adversarial training
EPS_EVAL    = [0.05, 0.1, 0.2]  # epsilon utk evaluasi robustness
ADV_RATIO   = 0.20     # proporsi adversarial dlm D_robust
MAX_SALIENCY = 40000   # batasi jumlah sampel utk komputasi saliency (efisiensi)

print('CIC pkl    :', os.path.exists(CIC_PKL))
print('UNSW train :', os.path.exists(UNSW_TRAIN))
print('UNSW test  :', os.path.exists(UNSW_TEST))

In [ ]:
# --- Mapping fitur Model A (nama kanonik <- CIC <- UNSW) ---
MAP_A = {
    'duration'  : ('Flow Duration',    'dur'),
    'fwd_pkts'  : ('Tot Fwd Pkts',     'spkts'),
    'bwd_pkts'  : ('Tot Bwd Pkts',     'dpkts'),
    'fwd_bytes' : ('TotLen Fwd Pkts',  'sbytes'),
    'bwd_bytes' : ('TotLen Bwd Pkts',  'dbytes'),
    'fwd_mean'  : ('Fwd Pkt Len Mean', 'smean'),
    'bwd_mean'  : ('Bwd Pkt Len Mean', 'dmean'),
    'src_load'  : ('Flow Byts/s',      'sload'),
    'dst_load'  : ('Bwd Pkts/s',       'dload'),
}
CANON = list(MAP_A.keys())
print('Model A:', CANON)

In [ ]:
# --- Muat CIC (un-scale) + label biner (Benign=0 -> 0; sisanya -> 1) ---
with open(CIC_PKL, 'rb') as f:
    d = pickle.load(f)
cic_feats = list(d['feature_names'])
X = np.asarray(d['X'], dtype=float)
scaler = d.get('scaler', None)
if scaler is not None and hasattr(scaler, 'scale_') and hasattr(scaler, 'mean_'):
    X_orig = X * scaler.scale_ + scaler.mean_
else:
    X_orig = X
cic_df = pd.DataFrame(X_orig, columns=cic_feats)

# label biner CIC — terverifikasi di T3: label_mapping Benign=0, sisanya attack
y_cic_raw = np.asarray(d['y'])
lm = d.get('label_mapping', None)
if lm is not None and 'Benign' in lm:
    benign_code = lm['Benign']
    y_cic_bin = (y_cic_raw != benign_code).astype(int)
    print(f"label_mapping ditemukan; Benign={benign_code}")
else:
    # fallback: asumsikan 0 = benign (dgn peringatan)
    y_cic_bin = (y_cic_raw != 0).astype(int)
    print('PERINGATAN: label_mapping tak ada; asumsi 0=Benign')
print('CIC:', cic_df.shape, '| biner:', dict(pd.Series(y_cic_bin).value_counts()))

In [ ]:
# --- Muat UNSW (train=175k, test=82k) + label biner ---
unsw_tr = pd.read_csv(UNSW_TRAIN)
unsw_te = pd.read_csv(UNSW_TEST)
y_unsw_tr = unsw_tr['label'].astype(int).values
y_unsw_te = unsw_te['label'].astype(int).values
print('UNSW train:', unsw_tr.shape, dict(pd.Series(y_unsw_tr).value_counts()))
print('UNSW test :', unsw_te.shape, dict(pd.Series(y_unsw_te).value_counts()))

In [ ]:
# --- Util: matriks fitur kanonik + preprocessing ---
def build_matrix(df, side):
    idx = 0 if side == 'cic' else 1
    cols = [MAP_A[c][idx] for c in CANON]
    out = df[cols].copy(); out.columns = CANON
    out = out.replace([np.inf, -np.inf], np.nan)
    out = out.fillna(out.median(numeric_only=True)).fillna(0.0)
    return out.astype(float).values

def make_xgb():
    return XGBClassifier(
        objective='binary:logistic', eval_metric='logloss',
        max_depth=8, learning_rate=0.1, n_estimators=200,
        subsample=0.8, colsample_bytree=0.8,
        n_jobs=-1, random_state=RANDOM_SEED, tree_method='hist')

def ev(y_true, y_pred):
    return dict(mcc=float(matthews_corrcoef(y_true, y_pred)),
                f1=float(f1_score(y_true, y_pred, zero_division=0)),
                acc=float(accuracy_score(y_true, y_pred)),
                confusion=confusion_matrix(y_true, y_pred).tolist())

In [ ]:
# --- Serangan: finite-difference saliency + FGSM (biner) ---
def compute_loss_bin(model, X, y_true):
    """binary cross-entropy per sampel."""
    p = model.predict_proba(X)[:, 1]
    p = np.clip(p, 1e-15, 1 - 1e-15)
    y = y_true.astype(float)
    return -(y * np.log(p) + (1 - y) * np.log(1 - p))

def saliency_fd(model, X, y_true, h=H_STEP):
    """central finite-difference gradient dL/dx_i (di ruang ter-z-score)."""
    n, m = X.shape
    S = np.zeros((n, m))
    for i in range(m):
        Xp = X.copy(); Xp[:, i] += h
        Xm = X.copy(); Xm[:, i] -= h
        S[:, i] = (compute_loss_bin(model, Xp, y_true) - compute_loss_bin(model, Xm, y_true)) / (2 * h)
    return S

def fgsm(X, S, eps):
    return X + eps * np.sign(S)

def make_adv(model, Xs, ys, eps, cap=MAX_SALIENCY):
    """bangkitkan adversarial pada subset (cap) sampel."""
    n = min(cap, len(Xs))
    idx = np.random.RandomState(RANDOM_SEED).choice(len(Xs), n, replace=False)
    Xsub, ysub = Xs[idx], ys[idx]
    S = saliency_fd(model, Xsub, ysub)
    return fgsm(Xsub, S, eps), ysub

In [ ]:
# --- Siapkan matriks + z-score per dataset ---
Xc = build_matrix(cic_df, 'cic')
Xu_tr_raw = build_matrix(unsw_tr, 'unsw')
Xu_te_raw = build_matrix(unsw_te, 'unsw')

# CIC: split internal (train/test) utk baseline in-domain
Xc_tr_raw, Xc_te_raw, yc_tr, yc_te = train_test_split(
    Xc, y_cic_bin, test_size=0.3, random_state=RANDOM_SEED, stratify=y_cic_bin)

sc_cic = StandardScaler().fit(Xc_tr_raw)
Xc_tr = sc_cic.transform(Xc_tr_raw); Xc_te = sc_cic.transform(Xc_te_raw)

sc_unsw = StandardScaler().fit(Xu_tr_raw)
Xu_tr = sc_unsw.transform(Xu_tr_raw); Xu_te = sc_unsw.transform(Xu_te_raw)

print('CIC  train/test:', Xc_tr.shape, Xc_te.shape)
print('UNSW train/test:', Xu_tr.shape, Xu_te.shape)

In [ ]:
# --- Fungsi: latih baseline + robust utk satu sumber, evaluasi 4 kondisi ---
def run_direction(name, X_src_tr, y_src_tr, X_src_te, y_src_te,
                  X_tgt_te, y_tgt_te, tgt_name):
    """Latih di src; uji di src (clean/adv) & tgt (clean/adv)."""
    print('='*72); print(f'ARAH: latih {name} -> uji {name} & {tgt_name}'); print('='*72)
    out = {}

    # --- baseline (clean training) ---
    base = make_xgb(); base.fit(X_src_tr, y_src_tr)

    # adversarial pada test src & tgt (thd model baseline) utk evaluasi
    def adv_eval_block(model, tag):
        b = {}
        b[f'{name}_clean'] = ev(y_src_te, model.predict(X_src_te))
        b[f'{tgt_name}_clean'] = ev(y_tgt_te, model.predict(X_tgt_te))
        for e in EPS_EVAL:
            Xa_s, ya_s = make_adv(model, X_src_te, y_src_te, e)
            b[f'{name}_adv_eps{e}'] = ev(ya_s, model.predict(Xa_s))
            Xa_t, ya_t = make_adv(model, X_tgt_te, y_tgt_te, e)
            b[f'{tgt_name}_adv_eps{e}'] = ev(ya_t, model.predict(Xa_t))
        return b

    out['baseline'] = adv_eval_block(base, 'baseline')

    # --- adversarial training: D_robust = clean UNION adv(train) ---
    Xa_tr, ya_tr = make_adv(base, X_src_tr, y_src_tr, EPS_TRAIN)
    n_clean = len(X_src_tr)
    n_adv_target = int(n_clean * ADV_RATIO / (1 - ADV_RATIO))  # adv/(clean+adv)=ratio
    n_adv = min(n_adv_target, len(Xa_tr))
    sel = np.random.RandomState(RANDOM_SEED).choice(len(Xa_tr), n_adv, replace=False)
    X_rob = np.vstack([X_src_tr, Xa_tr[sel]])
    y_rob = np.concatenate([y_src_tr, ya_tr[sel]])
    print(f'  D_robust: {n_clean:,} clean + {n_adv:,} adv = {len(X_rob):,}')

    rob = make_xgb(); rob.fit(X_rob, y_rob)
    out['robust'] = adv_eval_block(rob, 'robust')

    # cetak ringkas MCC
    def show(block, tag):
        print(f'  [{tag}]')
        for k, v in block.items():
            print(f'    {k:22s} MCC={v["mcc"]:+.4f}  F1={v["f1"]:.4f}  ACC={v["acc"]:.4f}')
    show(out['baseline'], 'BASELINE'); show(out['robust'], 'ROBUST')
    return out

In [ ]:
# --- Jalankan dua arah ---
t0 = time.time()
results = {}
# Arah 1: latih CIC -> uji CIC & UNSW(test)
results['train_cic'] = run_direction(
    'CIC', Xc_tr, yc_tr, Xc_te, yc_te, Xu_te, y_unsw_te, 'UNSW')
# Arah 2: latih UNSW(train) -> uji UNSW(test) & CIC(test)
results['train_unsw'] = run_direction(
    'UNSW', Xu_tr, y_unsw_tr, Xu_te, y_unsw_te, Xc_te, yc_te, 'CIC')
print(f'\nSelesai dalam {time.time()-t0:.1f}s')

In [ ]:
# --- Ringkasan fokus: apakah adversarial training menaikkan MCC cross-dataset? ---
def cross_summary():
    rows = []
    # arah CIC: cross = UNSW_clean
    b = results['train_cic']['baseline']['UNSW_clean']['mcc']
    r = results['train_cic']['robust']['UNSW_clean']['mcc']
    rows.append(dict(arah='latih CIC -> uji UNSW (clean)', baseline_mcc=round(b,4),
                     robust_mcc=round(r,4), delta=round(r-b,4)))
    # arah UNSW: cross = CIC_clean
    b = results['train_unsw']['baseline']['CIC_clean']['mcc']
    r = results['train_unsw']['robust']['CIC_clean']['mcc']
    rows.append(dict(arah='latih UNSW -> uji CIC (clean)', baseline_mcc=round(b,4),
                     robust_mcc=round(r,4), delta=round(r-b,4)))
    # in-domain robustness (eps=0.1) sbg konteks
    b = results['train_cic']['baseline']['baseline_adv_eps0.1']['mcc'] if 'baseline_adv_eps0.1' in results['train_cic']['baseline'] else None
    return pd.DataFrame(rows)

summ = cross_summary()
print('FOKUS PAPER Q1 — generalisasi lintas-jaringan (uji clean di dataset lain):')
print(summ.to_string(index=False))
print('\nInterpretasi: delta>0 = adversarial training MEMBANTU cross-network;')
print('delta~0 = tidak cukup (temuan: perlu mekanisme cross-network khusus).')

In [ ]:
# --- Simpan hasil ---
meta = dict(
    deskripsi='Adversarial training cross-dataset (Model A biner). finite-diff+FGSM ala Paper 1; z-score per dataset.',
    config=dict(model='A (9 fitur)', h_step=H_STEP, eps_train=EPS_TRAIN,
                eps_eval=EPS_EVAL, adv_ratio=ADV_RATIO, max_saliency=MAX_SALIENCY,
                xgb=dict(max_depth=8, lr=0.1, n_estimators=200, subsample=0.8, colsample=0.8)),
    features=CANON,
    cross_summary=summ.to_dict('records'),
    results=results,
)
with open(OUT_JSON, 'w') as f:
    json.dump(meta, f, indent=2)
print('Saved:', OUT_JSON)